# **Retrieval-Augmented Generation (RAG) Tutorial: CPU-Only with Ollama and ChromaDB**
*Using Housing Data from a CSV File*

---

## **1. Introduction to RAG**

**Retrieval-Augmented Generation (RAG)** is a technique that combines **information retrieval** (fetching relevant documents) with **text generation** (using a language model). It’s useful for building Q&A systems, chatbots, or search engines that provide accurate, context-aware answers.

### **Why RAG?**
- **Accuracy**: Grounds answers in real data, reducing hallucinations.
- **Flexibility**: Works with custom datasets (e.g., CSV, PDF, databases).
- **Efficiency**: Uses embeddings to quickly find relevant information.

---

## **2. How RAG Works**

### **Core Components**
1. **Document Loading**: Extract text from sources (e.g., CSV, PDF, web).
2. **Chunking**: Split text into smaller pieces for embedding.
3. **Embedding**: Convert text into numerical vectors using a model.
4. **Vector Storage**: Store embeddings in a database (e.g., ChromaDB).
5. **Retrieval**: Find the most relevant chunks for a query.
6. **Generation**: Use an LLM (e.g., Ollama) to generate answers from retrieved context.

---

## **3. Tools and Setup**

### **Prerequisites**
- **Python 3.8+**
- **Ollama** (for running local LLMs)
- **ChromaDB** (for vector storage)
- **Sentence Transformers** (for embeddings)
- **Pandas** (for CSV handling)

### **Installation**
```bash
pip install ollama chromadb sentence-transformers pandas
```
Download Ollama and pull a small model:
```bash
ollama pull mistral  # or tinyllama
```

---

## **4. Step-by-Step Implementation**

---


### **Step 1: Load and Preprocess the CSV**

In [1]:
import pandas as pd

# Load the CSV
df = pd.read_csv('data/books.csv')

# Combine columns into a single text field
df['content'] = df.apply(
    lambda row: f"Title: {row['title']}, Author: {row['author']}, Genre: {row['genre']}, Description: {row['description']}",
    axis=1
)
documents = df['content'].tolist()

### **Step 2: Generate Embeddings**

In [2]:
from sentence_transformers import SentenceTransformer

# Load a lightweight embedding model
model = SentenceTransformer('BAAI/bge-base-en-v1.5')

# Convert documents to embeddings
embeddings = model.encode(documents)


/home/fhakym/Documents/Github_projects/Experiments/LLMs/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Theory**:
- **Embeddings** are numerical representations of text.
- `all-MiniLM-L6-v2` is a small, efficient model for CPU.

### **Step 3: Store Embeddings in ChromaDB**

Chroma is a lightweight open-source vector-based database for AI applications.


In [3]:
import chromadb

# Initialize ChromaDB
client = chromadb.Client()
# Create a new collection, suppressing the previous one

try:
    collection = client.create_collection(name="housing")
except:
    client.delete_collection("housing")
    collection = client.create_collection(name="housing")

# Add embeddings and documents
ids = [str(i) for i in range(len(documents))]
collection.add(
    documents=documents,
    embeddings=embeddings.tolist(),
    ids=ids
)

**Why ChromaDB?**
- Lightweight, open-source vector database.
- Works on CPU and supports fast similarity search.

### **Step 4: Retrieve Relevant Documents**

In [15]:
def retrieve(query, k=5):
    
    #k corresponds to the number of documents to retrieve
    query_embedding = model.encode([query])
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k
    )
    return results['documents'][0]
    

**How Retrieval Works**:
- The query is embedded into the same vector space.
- ChromaDB finds the closest embeddings (cosine similarity).
### **Step 5: Generate Answers with Ollama**

In [23]:
import requests

def generate_answer(query, context):
    prompt =  f"""
    Context: {context}

    Instruction:
    Answer the question using the information from the provided context. Make complete sentences.
    If the answer is not in the context, say "I don’t have enough information."

    Question: {query}

    Answer:
    """
    response = requests.post(
        'http://localhost:11434/api/generate',
        json={
            'model': 'gemma3:1b',
            'prompt': prompt,
            'stream': False
        }
    )
    return response.json()['response']

**Why Ollama?**
- Runs LLMs locally (good for experiments).
- `gemma3:1b` is a small but capable model for CPU.

### **Step 7: Test the Pipeline**

In [32]:
query = "What is the title and the author of the dystopian novel and give a small description?"
retrieved_docs = retrieve(query, k=2)
answer = generate_answer(query, retrieved_docs)
print(answer)

The title of the dystopian novel is 1984, and the author is George Orwell. It is a dystopian novel about totalitarianism and surveillance in a future society.




## **5. Key Concepts Explained**

### **Embeddings**
- Convert text to vectors (e.g., "house" → `[0.1, -0.5, 0.8, ...]`).
- Similar text → similar vectors.

### **Vector Databases**
- Store and search embeddings efficiently.
- ChromaDB uses **approximate nearest neighbor (ANN)** for speed.

### **Prompt Engineering**
- The prompt combines **retrieved context** + **user query** for the LLM.
- Example:
  ```
  Context: [retrieved documents]
  Question: [user query]
  Answer:
  ```

---

## **6. Customization and Scaling**

### **For Larger Datasets**
- **Batch embedding**: Process documents in chunks.
- **Advanced vector DBs**: Use Weaviate or Qdrant for scalability.

### **Improving Accuracy**
- Use larger embedding models.
- Fine-tune the LLM on your domain.


## **8. Conclusion**
This tutorial showed how to:
1. Load and preprocess housing data from a CSV.
2. Generate embeddings with Sentence Transformers.
3. Store and retrieve embeddings using ChromaDB.
4. Generate answers with Ollama.

**Next Steps**:
- Experiment with bigger models for both embedding and answer generation.
- Use a real dataset (using scrapping for example).
- Deploy as a web app (e.g., Flask + streamlit).